# Classes e objetos

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Escrever uma classe com campos, um construtor e métodos, e criar objetos a partir dela com new.
- Explicar o que uma variável guarda quando se refere a um objeto, e o que muda quando duas variáveis compartilham um.
- Dar a uma classe um método toString e membros static compartilhados por todos os seus objetos.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Uma classe descreve um tipo de objeto
2. Construtores e this
3. Variáveis guardam referências
4. Métodos que usam os campos do objeto
5. toString: como um objeto é impresso
6. static: compartilhado pela classe toda
7. Erros comuns

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check(...) imprime ✓ ou ✗; os exercícios abaixo usam
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": esperado " + expected + ", veio " + actual);
    }
}

## 1. Uma classe descreve um tipo de objeto

Até aqui os seus valores eram números e texto. Uma **classe** permite criar o seu próprio tipo de valor. Ela descreve o que cada objeto **sabe** (os **campos**, variáveis dentro dele) e o que ele **faz** (os **métodos**). O `new` cria um objeto a partir da classe, e cada objeto tem a própria cópia dos campos. Use um ponto para alcançá-los: `a.count`, `a.add()`.

In [ ]:
class Counter {
    int count;          // um campo: cada Counter tem o seu

    void add() {        // um método: trabalha com o campo deste objeto
        count++;
    }
}

void main() {
    Counter a = new Counter();
    Counter b = new Counter();
    a.add();
    a.add();
    b.add();
    IO.println("a: " + a.count + ", b: " + b.count);
}

🧪 **Experimente:** acrescente um método `void reset()` que volta `count` para 0, e chame-o só em `a`.

## 2. Construtores e this

Um **construtor** roda quando o `new` cria o objeto, para dar os primeiros valores aos campos. Ele tem o nome da classe e nenhum tipo de retorno. Lá dentro, `this.name` é o campo, enquanto `name` sozinho é o parâmetro: `this.name = name;` copia um para o outro. Os slides acompanham `new Pet("Rex", 3)` na memória.

#### 🎞️ Criando um objeto, passo a passo

![o new reserva o objeto no heap, o construtor preenche os campos, rex recebe a referência; same = rex compartilha o objeto, então um aniversário por same aparece por rex](constructor-steps.pt-br.svg)

In [ ]:
class Pet {
    String name;
    int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    String describe() {
        return name + " (" + age + (age == 1 ? " ano)" : " anos)");
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    Pet mia = new Pet("Mia", 1);
    IO.println(rex.describe());
    IO.println(mia.describe());
}

🧪 **Experimente:** tire o `this.` de `this.name = name;` e rode: o campo fica `null`, porque você copiou o parâmetro nele mesmo. Depois tente `new Pet("Bob")` com um argumento só.

## 3. Variáveis guardam referências

Uma variável de um tipo de classe não guarda o objeto: ela guarda uma **referência**, uma seta para o objeto no heap. `Pet same = rex;` copia a seta, então as duas variáveis alcançam o mesmo pet, e uma mudança por uma aparece pela outra. Passar um objeto para um método também passa a seta: o método pode mudar o objeto de quem chamou.

In [ ]:
class Pet {
    String name;
    int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }
}

void birthday(Pet p) {
    p.age++;            // muda o objeto de quem chamou
}

void main() {
    Pet rex = new Pet("Rex", 3);
    Pet same = rex;
    same.age = 10;
    IO.println(rex.age);
    birthday(rex);
    IO.println(rex.age);
    Pet other = new Pet("Rex", 3);   // um objeto separado, mesmo com os mesmos valores
    other.age = 99;
    IO.println(rex.age);
}

🧪 **Experimente:** preveja os três números antes de rodar. Compare com um parâmetro `int`, que um método não consegue mudar para quem chamou.

### ✏️ Exercício: Mude um objeto por um método

Um método recebe uma cópia da referência, não do objeto. Faça `fill` pôr v na caixa que recebe: o Box de quem chamou muda também.

In [ ]:
class Box {
    int value = -1;
}

/** Fills a new box with v, then reads it back from the caller's side. */
int filledWith(int v) {
    Box mine = new Box();
    fill(mine, v);
    return mine.value;
}

void fill(Box box, int v) {
    // TODO: mude box.value
}

void main() {
    check("quem chamou vê 7", filledWith(7), 7);
    check("e depois 0", filledWith(0), 0);
}

<details>
<summary>Ver uma resposta</summary>

```java
class Box {
    int value = -1;
}

/** Fills a new box with v, then reads it back from the caller's side. */
int filledWith(int v) {
    Box mine = new Box();
    fill(mine, v);
    return mine.value;
}

void fill(Box box, int v) {
    box.value = v;
}

void main() {
    check("quem chamou vê 7", filledWith(7), 7);
    check("e depois 0", filledWith(0), 0);
}
```

</details>

## 4. Métodos que usam os campos do objeto

Dentro de um método, os campos **deste** objeto estão disponíveis pelo nome (`age`), e os campos de outro objeto da mesma classe, por meio dele (`other.age`). É assim que os objetos respondem perguntas sobre si mesmos, ou se comparam com outros.

In [ ]:
class Box {
    double width;
    double height;

    Box(double width, double height) {
        this.width = width;
        this.height = height;
    }

    double area() {
        return width * height;
    }

    boolean fitsInside(Box other) {
        return width <= other.width && height <= other.height;
    }
}

void main() {
    Box small = new Box(2, 3);
    Box big = new Box(5, 5);
    IO.println(small.area() + " " + big.area());
    IO.println(small.fitsInside(big) + " " + big.fitsInside(small));
}

🧪 **Experimente:** acrescente `boolean isSquare()`, verdadeiro quando a largura é igual à altura, e teste nas duas caixas.

### ✏️ Exercício: Mais velho que

Escreva `isOlderThan(Pet other)`: verdadeiro quando este pet é mais velho que o outro.

In [ ]:
class Pet {
    String name;
    int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    boolean isOlderThan(Pet other) {
        // TODO: compare a idade deste pet com other.age
        return false;
    }
}

void main() {
    Pet rex = new Pet("Rex", 5);
    Pet mia = new Pet("Mia", 2);
    check("Rex é mais velho que Mia", rex.isOlderThan(mia), true);
    check("Mia não é mais velha que Rex", mia.isOlderThan(rex), false);
    check("não é mais velho que ele mesmo", rex.isOlderThan(rex), false);
}

<details>
<summary>Ver uma resposta</summary>

```java
class Pet {
    String name;
    int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    boolean isOlderThan(Pet other) {
        return age > other.age;
    }
}

void main() {
    Pet rex = new Pet("Rex", 5);
    Pet mia = new Pet("Mia", 2);
    check("Rex é mais velho que Mia", rex.isOlderThan(mia), true);
    check("Mia não é mais velha que Rex", mia.isOlderThan(rex), false);
    check("não é mais velho que ele mesmo", rex.isOlderThan(rex), false);
}
```

</details>

## 5. toString: como um objeto é impresso

Imprimir um objeto chama o método `toString()` dele. Sem ele você vê o nome da classe e um código, como `Pet@1b6d3586`. Escreva o seu para escolher o texto. O `@Override` avisa o compilador que você está substituindo um método que toda classe herda, e ele reclama se o nome estiver errado.

In [ ]:
class Pet {
    private final String name;
    private final int age;

    Pet(String name, int age) {
        this.name = name;
        this.age = age;
    }

    @Override
    public String toString() {
        return name + ", " + age;
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    IO.println(rex);
    IO.println("Meu pet: " + rex);
}

🧪 **Experimente:** apague o método `toString` e rode: esse é o texto padrão. Depois escreva errado, `tostring`, com o `@Override` ainda em cima, e leia o erro.

## 6. static: compartilhado pela classe toda

Um campo `static` pertence à **classe**, não a cada objeto: existe uma cópia só, compartilhada por todos, alcançada pelo nome da classe (`Pet.count`). Um método `static` não trabalha com um objeto, então não tem `this`, como `Math.max(3, 7)` ou `Integer.parseInt("42")`.

In [ ]:
class Pet {
    static int count = 0;       // uma cópia para a classe toda
    String name;                // uma cópia por objeto

    Pet(String name) {
        this.name = name;
        count++;
    }

    static String plural(int n) {
        return n == 1 ? "pet" : "pets";
    }
}

void main() {
    new Pet("Rex");
    new Pet("Mia");
    IO.println(Pet.count + " " + Pet.plural(Pet.count));
    IO.println(Math.max(3, 7));
}

🧪 **Experimente:** tire o `static` de `count` e rode: agora cada pet conta só a si mesmo. Depois tente usar `name` dentro de `plural`: um método static não tem objeto de onde ler.

## 7. Erros comuns

- **Uma variável sem objeto:** `Pet p = null;` (ou um campo que nunca recebeu valor) e depois `p.age` para o programa com uma `NullPointerException`.
- **Esquecer o `this.`** no construtor, e o campo nunca recebe o valor (passo 2).
- **Achar que `=` copia o objeto:** ele copia a referência (passo 3).
- **Chamar um método de objeto pela classe:** `Pet.describe()` só funciona com métodos `static`.

In [ ]:
class Pet {
    String name;
}

void main() {
    Pet p = null;
    try {
        IO.println(p.name);
    } catch (NullPointerException e) {
        IO.println("p não aponta para nenhum objeto");
    }
    p = new Pet();
    IO.println("agora p.name é " + p.name);
}

🧪 **Experimente:** dê um valor inicial a `name` na classe (`String name = "sem nome";`) e rode de novo.

## Em resumo

Uma **classe** descreve um tipo de objeto: os **campos** (o que cada objeto sabe) e os **métodos** (o que ele faz). O `new` cria um objeto, e o **construtor** (em que `this.name` é o campo e `name` o parâmetro) dá os primeiros valores aos campos. Uma variável de um tipo de classe guarda uma **referência** ao objeto, não o objeto em si.

```java
class Pet {
    private final String name;
    private int age;
    static int count = 0;

Pet(String name, int age) {
        this.name = name;
        this.age = age;
        count++;
    }

void birthday() {
        age++;
    }

@Override
    public String toString() {
        return name + " (" + age + ")";
    }
}

void main() {
    Pet rex = new Pet("Rex", 3);
    Pet same = rex;
    same.birthday();
    IO.println(rex + ", " + Pet.count + " pet(s)");
}
```

Cada objeto tem a própria cópia dos campos; um campo `static` pertence à classe e é compartilhado por todos os objetos.

**Cuidado:** `Pet same = rex;` copia a referência, não o objeto. As duas variáveis agora mudam o mesmo pet.

## Leituras

- [Classes and Objects in Java](https://www.freecodecamp.org/news/the-java-handbook/#heading-classes-and-objects-in-java) (freeCodeCamp, The Java Handbook, 15 min): Classes e objetos, métodos, construtores, e getters e setters. Pare antes de herança: ela vem na próxima lição. O livro escreve System.out.println dentro de public static void main; no Sphinx, IO.println e void main() fazem o mesmo.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Sua primeira classe** · Fácil
- **Classe Retângulo** · Fácil
- **Livro com toString()** · Fácil
- **Gerador de IDs (static)** · Fácil